# Sankey Diagram with External Labels

This notebook demonstrates the CooccurrenceSankeyVisualizer with external labels, which is now the default behavior.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import sys
from pathlib import Path

# Add project root to path
project_root = str(Path(os.path.abspath('')).parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Set environment variables for DBPATH
os.environ['DB_PATH'] = '/lunarc/nobackup/projects/snic2020-6-41/carl/dev.db'

# Now you can import using the full module path
from scripts.sqlite_backend.db_main import EasyNerDBHandler

db = EasyNerDBHandler()

## Basic Example with External Labels (Default)

In [ ]:
from scripts.sqlite_backend.statistics.cooccurence_sankey_visualizer import (
    CooccurrenceSankeyVisualizer,
)

# Get cooccurrences from database
cooccurrences = db.co.get_cooccurrences(e1_txt_norm_like="malaria", min_npmi=0.21, max_npmi=1, min_fq_doc_level=5, limit=15, offset=0)

# Create visualizer instance with default settings (now uses external labels by default)
sankey_viz = CooccurrenceSankeyVisualizer(source_category_name="Disease", target_category_name="Phenomenon")

# Create and display the diagram directly from cooccurrences
sankey_viz.create_diagram_from_cooccurrences(
    cooccurrences=cooccurrences,
    title="Disease-Phenomenon Co-occurrence Network (External Labels)",
).customize_layout(
    height=800,
    margin=dict(t=60, l=80, r=80, b=20),  # Note the increased left and right margins for external labels
).display()

## Example with Internal Labels (Disabling External Labels)

In [ ]:
# Create visualizer instance with external labels disabled
sankey_viz_internal = CooccurrenceSankeyVisualizer(
    source_category_name="Disease",
    target_category_name="Phenomenon",
    use_external_labels=False,  # Explicitly disable external labels
)

# Use the same cooccurrence data as before
sankey_viz_internal.create_diagram_from_cooccurrences(
    cooccurrences=cooccurrences,
    title="Disease-Phenomenon Co-occurrence Network (Internal Labels)",
).customize_layout(
    height=800,
    # Standard margins are fine for internal labels
    margin=dict(t=60, l=20, r=20, b=20),
).display()

## Advanced Example with Custom External Label Styling

In [ ]:
# Get cooccurrences focused on respiratory diseases
respiratory_cooccurrences = db.co.get_cooccurrences(e1_txt_norm_like="%respiratory%", min_npmi=0.2, max_npmi=1, min_fq_doc_level=3, limit=15, offset=0)

# Create visualizer instance with customized external labels
sankey_viz_custom = CooccurrenceSankeyVisualizer(
    source_category_name="Disease",
    target_category_name="Phenomenon",
    source_base_color="rgb(220, 20, 60)",  # Crimson for diseases
    target_base_color="rgb(0, 139, 139)",  # Dark cyan for phenomena
    external_label_offset=30,              # Increase the offset
    external_label_font=dict(
        family="Courier New, monospace",    # Different font
        size=14,
        color="#444444",
    ),
)

# Create the diagram
sankey_viz_custom.create_diagram_from_cooccurrences(
    cooccurrences=respiratory_cooccurrences,
    title="Respiratory Disease Co-occurrence Network (Custom External Labels)",
).customize_layout(
    height=800,
    template="plotly_white",               # Use a different template
    margin=dict(t=60, l=100, r=100, b=20),  # Even more horizontal margin for labels
).display()

## Saving a Diagram with External Labels

In [ ]:
# Get another interesting set of cooccurrences
thunder_cooccurrences = db.co.get_cooccurrences(e2_txt_norm_like="thunder%", min_npmi=0.4, max_npmi=1, min_fq_doc_level=2, limit=10, offset=0)

# Create visualizer with the default settings (external labels enabled)
sankey_viz = CooccurrenceSankeyVisualizer()

# Create the diagram and save it
sankey_viz.create_diagram_from_cooccurrences(
    cooccurrences=thunder_cooccurrences,
    title="Thunder Phenomena Co-occurrence Network",
).customize_layout(
    height=600,
    margin=dict(t=60, l=80, r=80, b=20),
).display().save("../results/thunder_sankey_with_external_labels.png")

## Tips for Using External Labels

1. **Appropriate Margins**: External labels work best with increased left and right margins (e.g., 80-100px) to prevent labels from being cut off.

2. **Label Offset**: Use `external_label_offset` to adjust how far the labels appear from the nodes. Default is 20.

3. **Font Customization**: Use `external_label_font` to customize the appearance of your labels.

4. **Hover Information**: Even with external labels, the original node labels (with category names) are preserved in hover information.

5. **Disabling External Labels**: Set `use_external_labels=False` if you prefer traditional internal node labels.